In [1]:
import os
import sys
import subprocess
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from pyarrow import fs
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
import datetime
os.environ["HADOOP_CONF_DIR"] = "/path/to/hadoop/conf/"
os.environ["JAVA_HOME"] = "/path/to/java"
os.environ["HADOOP_HOME"] = "/path/to/hadoop"
os.environ["ARROW_LIBHDFS_DIR"] = "/path/to/hadoop/lib/"
os.environ["CLASSPATH"] = subprocess.check_output(
    "$HADOOP_HOME/bin/hadoop classpath --glob", shell=True
).decode("utf-8")

hdfs = fs.HadoopFileSystem(
    host="hdfs://your-hdfs-host", port=8020
)

# import pandas as pd
from tqdm import tqdm
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.simplefilter('ignore', ConvergenceWarning)
import sys

if not sys.warnoptions:
    import warnings

    warnings.simplefilter("ignore")
from tqdm import trange


2024-08-12 12:55:06,336 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Related function

In [2]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import STL
from prophet import Prophet
import pandas as pd
from pmdarima import auto_arima


def measure_simility(arr1, arr2):
    arr1 = arr1.reshape((-1,))
    arr2 = arr2.reshape((-1,))
    score = np.corrcoef(arr1, arr2)[0, 1]
    return score


def pick_data_train(vt, arrs, n):
    score = []
    for arr in arrs:
        score.append(measure_simility(vt, arr))
    score = np.array(score)
    return np.argsort(score)[-n:], score[np.argsort(score)[-n:]]


def create_data_total(df, month, year, lag, shift):
    x_knowns = df[df.index < year].iloc[:, 11 + month - lag - shift: 11 + month - shift].values.reshape((-1, lag))
    # y_known = df[df.index == year].iloc[0,11 + month-lag: 11 +month].values
    y_known = df[df.index == year].iloc[0, 11 + month - lag: 11 + month].values
    # y_known = df.iloc[-1, 11 + month - lag: 11 + month].values
    x_news = df[df.index < year].iloc[:, 11 + month - shift].values
    # print(month,year,y_known)
    return x_knowns, y_known, x_news


def create_data_cumsum(df, month, year, lag, shift):
    x_knowns = df[df.index < year].iloc[:, 11 + month - lag - shift: 11 + month - shift].cumsum(axis=1).values.reshape(
        (-1, lag))
    y_known = df[df.index == year].iloc[:, 11 + month - lag: 11 + month].cumsum(axis=1).iloc[0, :].values
    # y_known = df.iloc[[-1], 11 + month - lag: 11 + month].cumsum(axis=1).iloc[0, :].values
    x_news = df[df.index < year].iloc[:, 11 + month - lag - shift: 12 + month - shift].cumsum(axis=1).iloc[:, -1].values
    return x_knowns, y_known, x_news


def create_data_rolling(df, month, year, lag, shift, n_rolling):
    y_old = df[df.index == year].iloc[0, 11 + month - 1 - n_rolling + 1:11 + month - 1].sum()
    df = df.rolling(n_rolling, axis=1).mean()
    x_knowns = df[df.index < year].iloc[:, 11 + month - lag - shift: 11 + month - shift].values.reshape((-1, lag))
    y_known = df[df.index == year].iloc[0, 11 + month - lag: 11 + month].values
    x_news = df[df.index < year].iloc[:, 11 + month - shift].values
    return x_knowns, y_known, x_news, y_old


def predict_total(known_y, known_x, new_x, order):
    # known_y = known_y.reshape((-1,))
    # known_x = known_x.reshape((-1,))
    coefficients = np.polyfit(known_x, known_y, order)
    estimated_func = np.poly1d(coefficients)
    return float(estimated_func(new_x))


def predict_cumsum(known_y, known_x, new_x, order):
    # known_y = known_y.reshape((-1,))
    # known_x = known_x.reshape((-1,))
    coefficients = np.polyfit(known_x, known_y, order)
    estimated_func = np.poly1d(coefficients)
    return float(estimated_func(new_x))


def predict(df_total, year, month: int, lag: int, order1: int, order2: int, shift: int, n_max):
    x_knowns, y_known, x_news = create_data_total(df_total, month, year, lag, shift)
    max_scores, coeffs = pick_data_train(y_known, x_knowns, n_max)
    pre = []
    for max_score in max_scores:
        x_knowns, y_known, x_news = create_data_total(df_total, month, year, lag, shift)
        x_known = x_knowns[max_score]
        x_new = x_news[max_score]

        pred1 = predict_total(known_y=y_known,
                              known_x=x_known,
                              new_x=x_new, order=order1)
        pre.append(pred1)
        shift = 0
        x_knowns, y_known, x_news = create_data_cumsum(df_total, month, year, lag, shift)
        x_known = x_knowns[max_score]
        x_new = x_news[max_score]
        pred2 = predict_cumsum(known_y=y_known,
                               known_x=x_known,
                               new_x=x_new, order=order2) - y_known[-1]

        pre.append(pred2)
    # if (month==1)&(lag==12):
    #     return np.mean(pre),coeffs.mean()
    # else:
    # n_rolling = 2
    # x_knowns,y_known,x_news,y_old = create_data_rolling(df_total,month,year,lag,shift,n_rolling)
    # max_scores,coeffs = pick_data_train(y_known,x_knowns,n_max)
    # for max_score in max_scores:
    #     x_known = x_knowns[max_score]
    #     x_new = x_news[max_score]
    #
    #     pred3 = predict_total(known_y=y_known,
    #                           known_x=x_known,
    #                           new_x=x_new,order=order1)*n_rolling - y_old
    #     pre.append(pred3)
    return np.mean(pre), coeffs.mean()


def predict_master(data: pd.DataFrame, d_date: pd.DataFrame, month, year, lag, p, d, q, n_max, type_decomp,
                   methods_season, method_resid, method_trend):
    date = f"{year}-{month}-1"
    # print(date)
    # d_train = data[data.index < f"{year}-1-1"]
    d_train = data[data.index < date]
    d_train = pd.concat([d_train, d_date], axis=1).dropna(axis=0)

    d_residuals, d_trend, d_seasonal = detrend(d_train, type=type_decomp)
    # y_residual=0
    y_residual = forecast_residual(d_residuals.values, p, d, q, method_resid=method_resid)
    # except:
    #     y_residual = 0

    d_seasonal.name = name_province
    d_seasonal = pd.concat([d_seasonal, d_date], axis=1).fillna(0)
    d_seasonal = d_seasonal.pivot_table(columns='month', index='year', values=name_province)
    y_seasonal = forecast_seasonal(d_seasonal, month, year, lag, n_max, methods_season)

    d_trend.name = name_province
    d_trend = pd.concat([d_trend, d_date], axis=1).dropna()
    try:
        y_trend = forecast_trend(d_trend.iloc[-2, 0], d_trend.iloc[-1, 0], method_trend=method_trend)
    except:
        y_trend = 0

    # d_residuals = pd.concat([d_residuals, d_date], axis=1).dropna()
    # y_re3 = d_seasonal.pivot_table(columns='month', index='year', values=name_province).dropna(axis=0).mean(axis=0)[
    #     month]
    # d_residuals = d_residuals.pivot_table(columns='month', index='year', values=name_province)
    # d_trend = d_trend.pivot_table(columns='month', index='year', values=name_province)
    # y_re2 = forecast_seasonal(d_trend, month, year, lag, 2)
    # return y_lr + (y_re1 +y_re2+y_re)/3,y_re
    # return (y_re3 + y_re1) / 2 + y_re2
    return y_trend, y_seasonal, y_residual


def detrend(df: pd.DataFrame, type='STL'):
    if type == 'STL':
        data = df.copy()
        data = data.iloc[:, 0]
        stl = STL(data, period=12)
        res = stl.fit()
        trend = res.trend
        resid = res.resid
        season = res.seasonal
        return resid, trend, season
    if type == 'prophet':
        data = df.copy()
        data = data.iloc[:, [0]]
        data = data.reset_index()
        data.columns = ['ds', 'y']
        model = Prophet(yearly_seasonality=12)
        model.fit(data)
        forecaster = model.predict(model.make_future_dataframe(periods=0, freq='MS'))
        forecaster['y'] = data['y']
        forecaster['resid'] = forecaster.apply(lambda row: (row.y - (row.yearly + row.trend)), axis=1)
        forecaster = forecaster.set_index('ds')
        df_resid = forecaster['resid']
        df_trend = forecaster['trend']
        df_season = forecaster['yearly']
        df_season.name = 'seasonal'
        return df_resid, df_trend, df_season
    else:
        return None, None, None


def compute_error(data, month, year, y_pr):
    y_tr = data[data.index == f'{year}-{month}-1'].values[0]
    return (y_pr - y_tr) / y_tr, y_tr


def forecast_seasonal(data, month, year, lag, n_max, method='statistics'):
    if method == 'statistics':
        df = data.copy()
        df = pd.concat([df.shift(1, axis=0), df, df.shift(-1, axis=0)], axis=1).iloc[1:, :] + 10e10
        # lag = int(params[month - 1][1])
        # shift = int(params[month - 1][2])
        # order1 = int(params[month - 1][3])
        # order2 = int(params[month - 1][4])
        # lag = 6
        shift = 0
        order1 = 1
        order2 = 1
        y, _ = predict(df, year, month, lag, order1, order2, shift, n_max)
        return y - 10e10
    if method == 'average':
        df = data.copy()
        df = df[df.index < year]
        # print(df.dropna(axis=0).mean(axis=0))
        y = df.dropna(axis=0).mean(axis=0)[month]
        # print(y)
        return y
    if method == 'latest':
        df = data.copy()
        df = df[df.index < year]
        # print(df.dropna(axis=0).mean(axis=0))
        y = df.dropna(axis=0).iloc[-1, :][month]
        # print(y)
        return y
    else:
        raise "Not specify the season method"


def forecast_trend(y_1, y_t, method_trend='average'):
    if method_trend == 'average':
        y_pr_trend = y_t + (y_t - y_1)
        # y_pr_trend = y_t
        return y_pr_trend
    if method_trend == 'latest':
        return y_t
    else:
        raise "Not specify the trend method"


def forecast_residual(series, p, d, q, method_resid='arima'):
    if method_resid == 'arima':
        try:
            model = ARIMA(series, order=(p, d, q))
            model_fit = model.fit()
            return model_fit.forecast(freq='MS')[0]
        except:
            return 0
    # if method_resid == 'autoarima':
    #     model = auto_arima(series, seasonal=False, trace=False)
    #     return model.predict(n_periods=1, freq='MS')[0]
    else:
        return 0


def get_ytr(df, month, year):
    try:
        filter = f'{year}-{month}-1'
        return df[df.index == filter].values[0][0]
    except:
        return None


# Load data from 13 province

In [21]:
df_sl_13province = pd.read_parquet(
    "/path/to/company_data/sa/cpc/store/data/SUM_TOTAL/total_sl_13_provinces_2024-07.parquet",
    filesystem=hdfs,
)


In [22]:
df_sl_13province

,PHIEN,BDINH,DANANG,DLAC,DNONG,GLAI,KHOA,KTUM,PYEN,QBINH,QNAM,QNGAI,QTRI,TTHUE,TOTAL
0,2021-01-01,172740120,203601803,147457074,44596327,90428875,154108940,41738931,64427738,75663066,157055696,174093766,54194223,128704328,1508810887
1,2021-02-01,144247290,180774297,147913340,48214165,106511589,146572064,34211284,62135056,72019515,140222204,150764661,48705202,117046364,1399337031
2,2021-03-01,181550824,194145612,186440175,48782238,108250474,152211725,38802307,70020661,73264978,154367593,159436181,50696210,129905222,1547874200
3,2021-04-01,199746953,238971267,206295423,52172946,124368219,192014064,37639153,84095864,85638321,192248075,183784444,58072285,149330895,1804377909
4,2021-05-01,204394644,245267546,176763006,43851699,100472859,197897822,32011523,86273178,88530983,196539440,183387482,61960664,161610697,1778961543
5,2021-06-01,228505885,266731131,142944656,46174071,101666699,211777539,33170708,92921191,107204687,217478486,218354068,71719111,178211205,1916859437
6,2021-07-01,221138117,270969071,151337167,41506884,91817473,198623691,32665248,85375068,105889038,220382599,208500094,71489061,176280231,1875973742
7,2021-08-01,219844509,261542900,138773834,42523878,87648926,195990592,36277376,89138509,107068894,217134381,244827558,71962232,184218767,1896952356
8,2021-09-01,198720930,215147407,143631179,44390920,91234608,177671416,39612071,85693213,92221686,191223040,184386773,68179220,162030947,1694143410
9,2021-10-01,195872288,214121072,145116628,38845340,88811751,173821507,37360942,80852684,82113117,185683049,185432814,62444370,150443703,1640919265


In [23]:
# df_total_1 = df_sl_13province.copy().iloc[:, :-2]
df_total_1 = df_sl_13province.copy()
df_total_1["PHIEN"] = pd.to_datetime(df_total_1["PHIEN"])
df_total_1 = df_total_1.set_index('PHIEN')
df_total_1.sort_index(ascending=False)

,BDINH,DANANG,DLAC,DNONG,GLAI,KHOA,KTUM,PYEN,QBINH,QNAM,QNGAI,QTRI,TTHUE,TOTAL
PHIEN,,,,,,,,,,,,,,
2024-07-01,255772368,359420497,175574535,51863903,101472322,288388568,37254149,105210301,115404015,256516607,219072794,82544968,201949922,2250444949
2024-06-01,260833989,370651778,192202350,58677511,122640070,287084140,41322242,105866124,122718500,261090273,231255978,84618179,209714426,2348675560
2024-05-01,260856946,338171599,222915053,69759755,132602652,287016154,43327721,106753144,107833547,260995710,238125119,79620384,197414563,2345392347
2024-04-01,244363691,308628213,254755130,76584240,166302227,263096083,49888676,101403567,104588279,233382499,221977075,72590240,186712681,2284272601
2024-03-01,226434785,284965616,255725535,79694103,160766425,235694577,53305738,91294032,86277063,218984921,195208922,65839829,163365520,2117557066
2024-02-01,192574920,207447504,197532396,70362781,146878983,196068026,42041468,75196236,70169936,169600044,161678291,55613670,127720682,1712884937
2024-01-01,217752375,245424572,187220071,76774275,175083206,209372461,52703106,80418680,89993245,230284016,174027218,60340117,143831781,1943225123
2023-12-01,209589687,251941410,192772258,68487295,129279584,220393105,53503395,91446014,76917535,198279228,177979839,69479002,168910328,1908978680
2023-11-01,237220530,306060760,181575737,65081029,123236212,245231088,46903748,81903276,92552312,225265298,188298784,78325276,166285061,2037939111


In [33]:
pd.read_parquet('/path/to/company_data/user/<user>/cpc/data/fct_12t_sum_13province_test_2024.parquet',filesystem=hdfs)

,province,year,month,trend,season,resid,y_pr,y_tr,n_max,y_pr1,...,y_tr3,y_tr4,y_tr5,y_tr6,y_tr7,y_tr8,y_tr9,y_tr10,y_tr11,y_tr12
0,BDINH,2024,7,"[256878531.59585866, 261255998.7591457, 264014...","[18510762.207885742, 18981138.277160645, 16394...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[275389293.80374444, 280237137.0363064, 280408...","[255772368.0, nan, nan, nan, nan, nan, nan, na...",5,2.753893e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,DANANG,2024,7,"[331550957.85984576, 336196776.39362943, 33812...","[41150399.11402893, 51888595.59655762, 3535153...","[23480.37765097653, -1056016.8431415178, 15628...","[372724837.35152566, 387029355.14704555, 37363...","[359420497.0, nan, nan, nan, nan, nan, nan, na...",5,3.727248e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,DLAC,2024,7,"[216194230.3742893, 221906259.51403475, 225985...","[-15654948.216339111, -16349610.389038086, -50...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[200539282.1579502, 205556649.12499666, 220939...","[175574535.0, nan, nan, nan, nan, nan, nan, na...",5,2.005393e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,DNONG,2024,7,"[70299949.44034208, 71314476.38613176, 7232815...","[-6033527.270446777, -4170113.5448303223, -377...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[64266422.169895306, 67144362.84130144, 685521...","[51863903.0, nan, nan, nan, nan, nan, nan, nan...",5,6.426642e+07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,GLAI,2024,7,"[143401388.6645349, 147475347.30420658, 149891...","[-10515903.690231323, -18383708.78440857, -130...","[-582681.9743984524, -522826.06760597293, -381...","[132302802.99990512, 128568812.45219204, 13642...","[101472322.0, nan, nan, nan, nan, nan, nan, na...",5,1.323028e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,KHOA,2024,7,"[267736131.28110895, 269672684.2009511, 270161...","[20910439.675949097, 22605741.43978882, 139943...","[-391499.1319362108, -447655.43743396876, 4729...","[288255071.8251219, 291830770.20330596, 284628...","[288388568.0, nan, nan, nan, nan, nan, nan, na...",5,2.882551e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,KTUM,2024,7,"[49184338.480411425, 50123657.307048716, 50914...","[-6321331.688446045, -578237.4199066162, 12105...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[42863006.79196538, 49545419.8871421, 52125204...","[37254149.0, nan, nan, nan, nan, nan, nan, nan...",2,4.286301e+07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,PYEN,2024,7,"[103385625.75100623, 104270278.33472708, 10492...","[7803910.882492065, 9761824.77116394, 8066699....","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[111189536.6334983, 114032103.10589102, 112991...","[105210301.0, nan, nan, nan, nan, nan, nan, na...",2,1.111895e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,QBINH,2024,7,"[101388734.32781565, 102311050.38762332, 10401...","[22656887.7033844, 21936147.18600464, 11261044...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[124045622.03120005, 124247197.57362796, 11527...","[115404015.0, nan, nan, nan, nan, nan, nan, na...",2,1.240456e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,QNAM,2024,7,"[243948365.2911039, 248709096.33257955, 252745...","[26628797.46305847, 28034274.47265625, 1505081...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[270577162.7541624, 276743370.8052358, 2677958...","[256516607.0, nan, nan, nan, nan, nan, nan, na...",5,2.705772e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
df_total_1 = df_total_1[df_total_1.index<'2021-01-01'].sort_index(ascending=False)

In [10]:
# df_total_2 =pd.read_parquet('/path/to/company_data/sa/cpc/store/data/SUM_TOTAL/total_sl_13_provinces_2023-12.parquet',filesystem=hdfs).sort_values('PHIEN',ascending=False).set_index('PHIEN').iloc[:,:-1]

In [25]:
df_total = pd.concat([df_total_1,df_total_2],axis=0).sort_index(ascending=False)

In [11]:
df_total = df_total_1

# Pad none data to 2026-01-01

In [12]:
month_standing = df_total.index.max().month  # month is standing to forecast
df_total_padding = pd.DataFrame({col: None for col in df_total.columns},
                                index=pd.date_range(start=df_total.index.max() + datetime.timedelta(days=32), end=df_total.index.max() + datetime.timedelta(days=400), freq='MS'))
df_total_final = pd.concat([df_total, df_total_padding], axis=0)
# df_total_final

In [13]:
date_range = pd.date_range(start=df_total.index.min(), end=df_total.index.max() + datetime.timedelta(days=400), freq='MS')
df_datetime = pd.DataFrame({'stt': [i for i in range(date_range.shape[0])]}, index=date_range)
df_datetime['month'] = df_datetime.index.month
df_datetime['year'] = df_datetime.index.year

# Load parameters of 13 province

In [63]:
! hdfs dfs -ls /path/to/company_data/user/<user>/cpc/result

Found 8 items
drwxrwx---+  - <user> fdp          0 2024-01-11 21:09 /path/to/company_data/user/<user>/cpc/result/202309
-rwxrwx---+  3 <user> fdp      15315 2024-03-05 20:02 /path/to/company_data/user/<user>/cpc/result/fct_1t_sum_13province_20240305_v1.parquet
-rwxrwx---+  3 <user> fdp     228786 2024-03-05 20:15 /path/to/company_data/user/<user>/cpc/result/fct_gr1_12t_sum_13province_20240305_v2023.parquet
-rwxrwx---+  3 <user> fdp      15315 2024-03-05 20:13 /path/to/company_data/user/<user>/cpc/result/fct_gr1_1t_sum_13province_20240305_v2022.parquet
-rwxrwx---+  3 <user> fdp      15315 2024-03-05 20:14 /path/to/company_data/user/<user>/cpc/result/fct_gr1_1t_sum_13province_20240305_v2023.parquet
-rwxrwx---+  3 <user> fdp     231162 2024-03-05 21:10 /path/to/company_data/user/<user>/cpc/result/fct_gr2_12t_sum_13province_20240305_v2023.parquet
-rwxrwx---+  3 <user> fdp      15355 2024-03-05 20:12 /path/to/company_data/user/<user>/cpc/result/fct_gr2_1t_sum_13province_20240305_v2022.parqu

In [14]:
dfp = pd.read_parquet(
    '/path/to/company_data/user/<user>/cpc/data/parameter/final_params_13province_u2022_v2.parquet',
    filesystem=hdfs)

# Forecast

In [16]:
lst_pr = df_total_final.columns
df_result = pd.DataFrame()
years = [2024]
for n_province in trange(len(lst_pr)):
# for n_province in [1]:
    # df = df_total.iloc[:, [n_province]]
    name_province = lst_pr[n_province]
    try:
        dp = dfp[dfp.province == name_province]
        lag = 6
        # n_max = 2
        n_max = dp.n_max.values[0]
        # type_decomp = dp.type_decomp.values[0]
        # method_season =dp.method_season.values[0]
        method_trend = dp.method_trend.values[0]
        method_resid = dp.method_resid.values[0]
        # n_max = 5
        type_decomp = 'STL'
        method_season = 'statistics'
        p = 1
        d = 0
        q = 1
        # df = df.sort_index()
        # print(name_province)
    except:
        print(f'not filter name {name_province}')
        lag = 6
        n_max = 5
        method_trend = 'average'
        method_season = 'statistics'
        type_decomp = 'STL'
        method_resid = 'arima'
        p = 1
        d = 0
        q = 1

    for year in years:
        for month in [8]:
            # print(month)
            if (month==4)&(year==2024):
                break
            # try:
            # print(month)
            lst_ypr = []
            lst_ytr = []
            lst_trend = []
            lst_resid = []
            lst_season = []
            df = df_total_final.copy().iloc[:, [n_province]]
            df = df.sort_index()
            for t in range(12):
                # try:
                # print(t)
                if (month + t > 12):
                    year = year + 1
                    month = month - 12
                    # print(month)

                    y_trend, y_seasonal, y_residual = predict_master(
                        data=df,
                        d_date=df_datetime,
                        month=month+t, year=year,
                        lag=lag, p=p, d=d, q=q,
                        n_max=n_max,
                        type_decomp=type_decomp,
                        methods_season=method_season, method_resid=method_resid,
                        method_trend=method_trend)
                    y_tr = get_ytr(df, month + t, year)
                    y_pr = y_trend + y_seasonal + y_residual

                    df.at[f"{year}-{month + t}-1", name_province] = y_pr
                    year = year - 1
                    month = month + 12
                else:
                    y_trend, y_seasonal, y_residual = predict_master(
                        data=df,
                        d_date=df_datetime,
                        month=month + t, year=year,
                        lag=lag, p=p, d=d, q=q,
                        n_max=n_max,
                        type_decomp=type_decomp,
                        methods_season=method_season, method_resid=method_resid,
                        method_trend=method_trend)

                    y_tr = get_ytr(df, month + t, year)
                    y_pr = y_trend + y_seasonal + y_residual

                    df.at[f"{year}-{month + t}-1", name_province] = y_pr

                lst_ypr.append(y_pr)
                lst_ytr.append(y_tr)
                lst_trend.append(y_trend)
                lst_season.append(y_seasonal)
                lst_resid.append(y_residual)

            # except:
            #
            #     print(year,month,t)
            #     break
            df_result = pd.concat([df_result, pd.DataFrame({
                'province': [name_province],
                'year': [year],
                'month': [month],
                'trend': [lst_trend],
                'season': [lst_season],
                'resid': [lst_resid],
                'y_pr': [lst_ypr],
                'y_tr': [lst_ytr],
                'n_max': [n_max],
            })])
        # except:break


 93%|█████████▎| 13/14 [00:08<00:00,  1.53it/s]

not filter name TOTAL


100%|██████████| 14/14 [00:09<00:00,  1.50it/s]


In [17]:
def expand_ypr(row):
    row = row['y_pr']
    return row[0], row[1], row[2], row[3], row[4], row[5], row[6], row[7], row[8], row[9], row[10], row[11]


def expand_ytr(row):
    row = row['y_tr']
    return row[0], row[1], row[2], row[3], row[4], row[5], row[6], row[7], row[8], row[9], row[10], row[11]


df_result[['y_pr1', 'y_pr2', 'y_pr3', 'y_pr4', 'y_pr5', 'y_pr6', 'y_pr7', 'y_pr8', 'y_pr9', 'y_pr10', 'y_pr11',
           'y_pr12', ]] = df_result.apply(expand_ypr, axis=1, result_type='expand')
df_result[['y_tr1', 'y_tr2', 'y_tr3', 'y_tr4', 'y_tr5', 'y_tr6', 'y_tr7', 'y_tr8', 'y_tr9', 'y_tr10', 'y_tr11',
           'y_tr12', ]] = df_result.apply(expand_ytr, axis=1, result_type='expand')

In [19]:
# df_result_1 = df_result[['province','month','year','y_pr1','y_tr1']]
# df_result_1['error'] = df_result_1.apply(lambda x: (x.y_pr1 - x.y_tr1 )/(x.y_tr1) ,axis=1)

In [72]:
df_result_1.pivot_table(index='province',columns=['year','month'],values='error')

year          2022                                                    \
month           1         2         3         4         5         6    
province                                                               
BDINH     0.027997  0.057070 -0.066132 -0.024932 -0.037646  0.040546   
DANANG   -0.075898  0.051846 -0.059835 -0.113588 -0.056589 -0.076570   
DLAC     -0.450252 -0.292921 -0.143782 -0.094466  0.999907  0.097389   
DNONG    -0.024476  0.594467  0.175513  0.328642  0.876939  0.580984   
GLAI     -0.102650 -0.056400  0.206102 -0.180565 -0.353405 -0.253534   
KHOA     -0.204075 -0.084181 -0.119069 -0.174844 -0.090581 -0.131746   
KTUM      0.132446  0.206339  0.002127 -0.001824 -0.339905 -0.223941   
PYEN      0.134300  0.063329 -0.012311 -0.068612 -0.106063 -0.139564   
QBINH    -0.094659  0.389992 -0.010315 -0.082739  0.073223  0.061103   
QNAM     -0.076130  0.015406 -0.011425 -0.059435 -0.055539 -0.033874   
QNGAI     0.005066  0.145247  0.140442  0.100155 -0.063467  0.380104   
QTRI      0.158032  0.007102  0.051278 -0.171308 -0.151782 -0.088494   
TTHUE    -0.181323  0.025615  0.040938 -0.037503 -0.031950  0.011751   

year                                              ...      2023            \
month           7         8         9         10  ...        5         6    
province                                          ...                       
BDINH     0.111660  0.154000  0.025221  0.024279  ... -0.030048 -0.037820   
DANANG    0.020310 -0.050732  0.019089 -0.011397  ... -0.018423  0.008950   
DLAC      0.101519 -0.179659 -0.037362  0.393854  ...  0.046492 -0.436756   
DNONG    -0.139524 -0.232014 -0.549747 -0.565445  ... -0.078204  0.314681   
GLAI     -0.227058 -0.143458 -0.352530 -0.178045  ...  0.041046 -0.025064   
KHOA     -0.060066  0.004510  0.116633  0.013111  ... -0.069400 -0.106530   
KTUM     -0.128783 -0.243328  0.003770 -0.060733  ... -0.032854 -0.012622   
PYEN     -0.187707 -0.058548  0.054202  0.021774  ...  0.112452  0.080980   
QBINH    -0.039991  0.038577 -0.021043  0.027094  ...  0.114220  0.293806   
QNAM     -0.071091 -0.092618 -0.037797 -0.082385  ...  0.009620  0.094931   
QNGAI     0.061429  0.087100 -0.048713  0.101880  ... -0.082559  0.061375   
QTRI     -0.030040 -0.059024 -0.026277 -0.095306  ...  0.182704  0.217827   
TTHUE     0.003816  0.024321  0.067442  0.033585  ... -0.042006 -0.014551   

year                                                                  \
month           7         8         9         10        11        12   
province                                                               
BDINH    -0.102714 -0.094402 -0.116252 -0.093319 -0.025000 -0.133640   
DANANG   -0.003880 -0.062402 -0.006483  0.102764  0.025045 -0.052446   
DLAC     -0.001912 -0.389322  0.187190 -0.091355 -0.007314  0.080320   
DNONG    -0.063175 -0.045579  0.678009  0.022463  0.219245 -0.040705   
GLAI     -0.155149 -0.332535 -0.181208 -0.056776  0.000684  0.102696   
KHOA      0.036176  0.007661  0.037574  0.160825  0.112636 -0.025487   
KTUM     -0.002307 -0.115490  0.132145 -0.010396  0.340272  0.122085   
PYEN     -0.208960 -0.249305 -0.176152 -0.081348  0.152205 -0.036262   
QBINH     0.122557  0.042811  0.261685 -0.132418 -0.041415  0.026484   
QNAM      0.083385  0.000290  0.038432  0.003001  0.043941  0.015629   
QNGAI    -0.058720 -0.008109  0.024898  0.105217  0.150885  0.148245   
QTRI      0.166294  0.141834 -0.171255 -0.202632 -0.126394 -0.070454   
TTHUE     0.067731  0.061179  0.046403 -0.097542  0.084434 -0.055004   

year          2024            
month           1         2   
province                      
BDINH    -0.213796  0.240258  
DANANG   -0.041141  0.194187  
DLAC     -0.018537  0.717112  
DNONG     0.143228  0.080936  
GLAI     -0.239608  0.203233  
KHOA     -0.000859  0.141014  
KTUM     -0.367903  0.292987  
PYEN     -0.235298  0.304847  
QBINH    -0.288753 -0.040528  
QNAM     -0.052209  0.150601  
QNGAI     0.045647 -0.000471  
QTRI     -0.133582  0.42348

In [93]:
df_result.loc[(df_result['year']==2024)&(df_result['month']==1),['province','year','month','y_pr1', 'y_pr2', 'y_pr3', 'y_pr4', 'y_pr5', 'y_pr6', 'y_pr7', 'y_pr8', 'y_pr9', 'y_pr10', 'y_pr11',
           'y_pr12',]].to_parquet('/path/to/company_data/user/<user>/cpc/result/fct_gr1_12t_sum_13province_20240305_v2022.parquet',filesystem=hdfs)

In [61]:
df_result[['province','year','month','y_pr1', 'y_pr2', 'y_pr3', 'y_pr4', 'y_pr5', 'y_pr6', 'y_pr7', 'y_pr8', 'y_pr9', 'y_pr10', 'y_pr11',
           'y_pr12',]].to_parquet('/path/to/company_data/user/<user>/cpc/result/fct_gr2_12t_sum_13province_20240305_v2022.parquet',filesystem=hdfs)

In [41]:
df_result.loc[(df_result.year==2024) & (df_result.month==1)].drop(columns=['trend','season','resid','y_pr','y_tr','n_max']).to_parquet('/path/to/company_data/user/<user>/cpc/predict/sum_13pr_t12_012024.parquet',filesystem=hdfs)

In [32]:
df_result.to_parquet('/path/to/company_data/user/<user>/cpc/result/202309/fct_12t_sum_13province_012022_012024.parquet',filesystem=hdfs,index=False)

In [53]:
df_result = pd.read_parquet('/path/to/company_data/user/<user>/cpc/predict/sum_13pr_t12_012024.parquet',filesystem=hdfs)
# df_result.to_excel('1.xlsx',index=False)
df_result

,province,year,month,y_pr1,y_pr2,y_pr3,y_pr4,y_pr5,y_pr6,y_pr7,...,y_tr3,y_tr4,y_tr5,y_tr6,y_tr7,y_tr8,y_tr9,y_tr10,y_tr11,y_tr12
0,BDINH,2024,1,2.081908e+08,2.098750e+08,2.233447e+08,2.472291e+08,2.568774e+08,2.755798e+08,2.789185e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,DANANG,2024,1,2.780267e+08,2.679979e+08,2.808798e+08,3.205148e+08,3.324893e+08,3.825997e+08,3.885532e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,DLAC,2024,1,1.790921e+08,2.077420e+08,2.276017e+08,2.270925e+08,2.102965e+08,1.964129e+08,1.902398e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,DNONG,2024,1,6.596548e+07,6.864852e+07,7.310554e+07,7.222412e+07,6.742478e+07,6.430713e+07,6.327455e+07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,GLAI,2024,1,1.184749e+08,1.429638e+08,1.436524e+08,1.466515e+08,1.324026e+08,1.274842e+08,1.230882e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,KHOA,2024,1,2.279067e+08,2.299983e+08,2.308905e+08,2.700482e+08,2.796654e+08,3.041809e+08,2.960533e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,KTUM,2024,1,4.408509e+07,4.726726e+07,4.854842e+07,4.651157e+07,4.136476e+07,4.169024e+07,4.166714e+07,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,PYEN,2024,1,8.405865e+07,8.555435e+07,8.727396e+07,1.047000e+08,1.068250e+08,1.137925e+08,1.138877e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,QBINH,2024,1,7.652408e+07,8.002257e+07,7.961700e+07,9.043588e+07,9.163720e+07,1.108054e+08,1.123508e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
0,QNAM,2024,1,2.000998e+08,1.937196e+08,2.059526e+08,2.260527e+08,2.317632e+08,2.509927e+08,2.576002e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
df_result

,province,year,month,trend,season,resid,y_pr,y_tr,n_max,y_pr1,...,y_tr3,y_tr4,y_tr5,y_tr6,y_tr7,y_tr8,y_tr9,y_tr10,y_tr11,y_tr12
0,BDINH,2024,8,"[257876890.28880456, 260309750.6574449, 259731...","[20217344.147140503, 15775400.629974365, 49172...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[278094234.43594503, 276085151.28741926, 26022...","[None, None, None, None, None, None, None, Non...",5,2.780942e+08,...,None,None,None,None,None,None,None,None,None,None
0,DANANG,2024,8,"[333905040.6943711, 335706148.824753, 33301547...","[52633077.08351135, 34972714.96255493, 2174971...","[-443794.7362708006, -320161.9775342429, 45584...","[386094323.0416117, 370358701.8097737, 3552210...","[None, None, None, None, None, None, None, Non...",5,3.860943e+08,...,None,None,None,None,None,None,None,None,None,None
0,DLAC,2024,8,"[214715291.86701596, 217786106.99800906, 22033...","[-15251812.981704712, -5375586.6100616455, -19...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[199463478.88531125, 212410520.3879474, 200634...","[None, None, None, None, None, None, None, Non...",2,1.994635e+08,...,None,None,None,None,None,None,None,None,None,None
0,DNONG,2024,8,"[70083093.54223573, 70758657.38116495, 7270025...","[-3451266.5691833496, -3859263.636352539, -950...","[-1667669.9287516559, 991175.1736062499, 35190...","[64964157.04430073, 67890568.91841866, 6354766...","[None, None, None, None, None, None, None, Non...",2,6.496416e+07,...,None,None,None,None,None,None,None,None,None,None
0,GLAI,2024,8,"[144584369.43575606, 146698163.2969375, 148393...","[-17390907.173034668, -14096110.69116211, -258...","[-787573.9689720881, -316343.75444766396, -279...","[126405888.2937493, 132285708.85132772, 122275...","[None, None, None, None, None, None, None, Non...",2,1.264059e+08,...,None,None,None,None,None,None,None,None,None,None
0,KHOA,2024,8,"[269693492.5613009, 270183541.82907724, 269399...","[22594384.682739258, 13996603.894317627, 15408...","[-447994.7054755275, 477394.0854986531, 10359....","[291839882.5385646, 284657539.8088935, 2848187...","[None, None, None, None, None, None, None, Non...",5,2.918399e+08,...,None,None,None,None,None,None,None,None,None,None
0,KTUM,2024,8,"[49157504.96170257, 49913691.0932785, 50887450...","[112616.72538757324, 1411471.812652588, 57502....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[49270121.68709014, 51325162.905931085, 509449...","[None, None, None, None, None, None, None, Non...",2,4.927012e+07,...,None,None,None,None,None,None,None,None,None,None
0,PYEN,2024,8,"[103240326.65027595, 103800575.53246816, 10410...","[10170106.718597412, 7891070.738861084, 731527...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[113410433.36887336, 111691646.27132924, 11142...","[None, None, None, None, None, None, None, Non...",2,1.134104e+08,...,None,None,None,None,None,None,None,None,None,None
0,QBINH,2024,8,"[100964062.93071057, 102533812.70711988, 10406...","[22359216.074508667, 11141621.446884155, -8779...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[123323279.00521924, 113675434.15400404, 10319...","[None, None, None, None, None, None, None, Non...",2,1.233233e+08,...,None,None,None,None,None,None,None,None,None,None
0,QNAM,2024,8,"[246287098.94512329, 250082693.2463964, 250761...","[28880507.77168274, 14790634.87223816, -475817...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[275167606.71680605, 264873328.11863455, 24600...","[None, None, None, None, None, None, None, Non...",5,2.751676e+08,...,None,None,None,None,None,None,None,None,None,None


In [107]:
df_gr2 = pd.read_parquet('/path/to/company_data/user/<user>/cpc/result/fct_gr2_12t_sum_13province_20240305_v2022.parquet',filesystem=hdfs)
df_gr1 = pd.read_parquet('/path/to/company_data/user/<user>/cpc/result/fct_gr1_12t_sum_13province_20240305_v2022.parquet',filesystem=hdfs)
df_gr = pd.read_parquet('/path/to/company_data/user/<user>/cpc/predict/sum_13pr_t12_012024.parquet',filesystem=hdfs)

In [109]:
df_tr1 = pd.read_parquet('/path/to/company_data/sa/cpc/store/data/SUM_TOTAL/group1_2024-02.parquet',filesystem=hdfs)
df_tr2 = pd.read_parquet('/path/to/company_data/sa/cpc/store/data/SUM_TOTAL/group2_2024-02.parquet',filesystem=hdfs)


In [113]:
df_tr2.sort_values('PHIEN',ascending=False).set_index('PHIEN').pivot_table(columns='PHIEN')

PHIEN,2014-01,2014-02,2014-03,2014-04,2014-05,2014-06,2014-07,2014-08,2014-09,2014-10,...,2023-05,2023-06,2023-07,2023-08,2023-09,2023-10,2023-11,2023-12,2024-01,2024-02
BDINH,85676223.0,75080708.0,79465304.0,95116905.0,97930607.0,114636319.0,105783137.0,107055955.0,119237769.0,97092451.0,...,161517745.0,174371090.0,175296344.0,181116818.0,182927470.0,165281816.0,175280557.0,138158499.0,147229609.0,147151741.0
DANANG,100118360.0,88047756.0,101774342.0,117513370.0,125469391.0,154086058.0,153852888.0,143879061.0,140726720.0,126183447.0,...,200999520.0,238746930.0,235229759.0,246441462.0,244251837.0,243215731.0,225414998.0,171819445.0,165551924.0,150103083.0
DLAC,73658187.0,91386769.0,98584105.0,101888144.0,75263491.0,78932096.0,70056906.0,70636475.0,75308700.0,80696164.0,...,162807024.0,138661755.0,200185631.0,133828810.0,145208622.0,148058992.0,150043322.0,160130832.0,154165944.0,179121993.0
DNONG,32284970.0,25396260.0,26596912.0,25964096.0,22166931.0,23222378.0,22019975.0,21046599.0,21962712.0,22116301.0,...,47124935.0,41689748.0,40002983.0,41127951.0,41462438.0,42328154.0,51212411.0,52904732.0,63476512.0,59341324.0
GLAI,53971412.0,60155435.0,64178401.0,66366784.0,55775137.0,56901498.0,52197045.0,50667430.0,53929748.0,50377194.0,...,108455257.0,98958514.0,91435517.0,92085527.0,95468275.0,89931641.0,107921901.0,114235539.0,158941946.0,135738741.0
KHOA,570430414.0,73991230.0,74191698.0,92389727.0,97671709.0,107771630.0,106231304.0,109377344.0,109192373.0,101334599.0,...,177744006.0,188867106.0,185219338.0,193358120.0,191386891.0,206154593.0,185299770.0,159931866.0,146747835.0,143689382.0
KTUM,16854571.0,17427190.0,17670131.0,19118961.0,18072299.0,19388706.0,17394606.0,16698382.0,17469790.0,17287037.0,...,33694943.0,31902747.0,29932251.0,30254898.0,31762247.0,31779668.0,35919086.0,39284758.0,34971390.0,33214068.0
PYEN,32204150.0,36393316.0,36029189.0,44836764.0,47714969.0,55074353.0,47180635.0,49687084.0,50072010.0,46374746.0,...,83623179.0,87523384.0,88042644.0,91025324.0,88049819.0,94597339.0,70246093.0,79288848.0,67499776.0,66998705.0
QBINH,28724740.0,30821950.0,30208391.0,36197130.0,40417584.0,52200996.0,48226510.0,48776443.0,46881013.0,42749078.0,...,78036883.0,95249684.0,97094847.0,93345710.0,85883342.0,74586249.0,79010192.0,62818211.0,77288431.0,61097831.0
QNAM,57607471.0,56684818.0,58226570.0,65981796.0,71626847.0,82504171.0,78860256.0,82114346.0,76649288.0,71794958.0,...,156591401.0,167422994.0,173356431.0,175728284.0,174879450.0,151553806.0,163603867.0,136837673.0,168311219.0,121578239.0


In [112]:
df_gr2

,province,year,month,y_pr1,y_pr2,y_pr3,y_pr4,y_pr5,y_pr6,y_pr7,y_pr8,y_pr9,y_pr10,y_pr11,y_pr12
0,BDINH,2024,1,1.466263e+08,1.514510e+08,1.549122e+08,1.821375e+08,1.883832e+08,2.051550e+08,2.044546e+08,2.075033e+08,2.096402e+08,1.971259e+08,1.925731e+08,1.807369e+08
0,DANANG,2024,1,2.042991e+08,1.969901e+08,2.038212e+08,2.364752e+08,2.496044e+08,2.916417e+08,2.970493e+08,3.021611e+08,2.962129e+08,2.787803e+08,2.662550e+08,2.535293e+08
0,DLAC,2024,1,1.530407e+08,1.764561e+08,1.863035e+08,1.973505e+08,1.844238e+08,1.702826e+08,1.975836e+08,1.565104e+08,1.676928e+08,1.658517e+08,1.673854e+08,1.724395e+08
0,DNONG,2024,1,5.238689e+07,5.867869e+07,6.134274e+07,6.419982e+07,6.074051e+07,5.883712e+07,5.848549e+07,6.165584e+07,6.321062e+07,6.475980e+07,7.206202e+07,1.281002e+08
0,GLAI,2024,1,1.113303e+08,1.311460e+08,1.292706e+08,1.351187e+08,1.255375e+08,1.228596e+08,1.195833e+08,1.189831e+08,1.242016e+08,1.192770e+08,1.315910e+08,1.363036e+08
0,KHOA,2024,1,1.604584e+08,1.658060e+08,1.635516e+08,1.949997e+08,2.029168e+08,2.176249e+08,2.130513e+08,2.165882e+08,2.147816e+08,2.072015e+08,1.950563e+08,1.777697e+08
0,KTUM,2024,1,3.410193e+07,3.810164e+07,3.696345e+07,4.185165e+07,4.066445e+07,3.999729e+07,3.905616e+07,3.920522e+07,4.102886e+07,3.982063e+07,4.302137e+07,4.477994e+07
0,PYEN,2024,1,7.295737e+07,7.991301e+07,7.590002e+07,9.482535e+07,9.875052e+07,1.057436e+08,1.056626e+08,1.079818e+08,1.065567e+08,1.054230e+08,9.503813e+07,9.507342e+07
0,QBINH,2024,1,7.152637e+07,7.420630e+07,6.794206e+07,7.764088e+07,8.486550e+07,9.241499e+07,1.021405e+08,1.008737e+08,9.471251e+07,8.187805e+07,7.966478e+07,7.140836e+07
0,QNAM,2024,1,1.407868e+08,1.388966e+08,1.420062e+08,1.611931e+08,1.682850e+08,1.841399e+08,1.868567e+08,1.882964e+08,1.821945e+08,1.712478e+08,1.659934e+08,1.627510e+08
